# Verification: Qwen2.5-0.5B Classification Head + LoRA (Final)

**Purpose:** Canonical verification of decoder + classification head + LoRA.
This is the clean version with all audit issues fixed.

**Model:** Qwen/Qwen2.5-0.5B (Sep 2024, Apache 2.0)
**Hardware:** Requires CUDA GPU (T4 or better).
**Expected time:** ~54 min for 3 epochs on T4, ~16 min on A100.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "accelerate", "scikit-learn",
    "peft", "tqdm",
])

0

In [2]:
import time
import random
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score

REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, LABEL_TO_ID, ID_TO_LABEL,
)

Cloning into '/tmp/course'...


## Configuration

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
NUM_EPOCHS = 3
BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 1
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 32
SEED = 42
MAX_SEQ_LENGTH = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
cc = torch.cuda.get_device_capability()
print(f"Device: {device}")
print(f"GPU: {gpu_name} ({vram:.1f} GB, cc {cc[0]}.{cc[1]})")

# Use bf16 on Ampere+ (cc >= 8.0), fp16 on Turing (cc 7.x)
AMP_DTYPE = "bf16" if cc[0] >= 8 else "fp16"

print(f"AMP: {AMP_DTYPE}")
print(f"Config: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}")

Device: cuda
GPU: Tesla T4 (14.6 GB, cc 7.5)
AMP: fp16
Config: 3 epochs, batch=32, lr=0.0002
LoRA: rank=16, alpha=32


## Load data

In [4]:
print("Loading canonical dataset...")
train_ds, val_ds, _ = load_course_data()
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Classes: {NUM_LABELS}")

Loading canonical dataset...


README.md:   0%|          | 0.00/606 [00:00<?, ?B/s]

data/train-00000-of-00001-f6ccf6490f1232(…):   0%|          | 0.00/10.2M [00:00<?, ?B/s]

data/test-00000-of-00001-0b18a4667c07409(…):   0%|          | 0.00/3.39M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/64292 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/6430 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6430 [00:00<?, ? examples/s]

Map:   0%|          | 0/21439 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21439 [00:00<?, ? examples/s]

Train: 57,846  Val: 6,430  Classes: 113


## Load model — no quantization, straight FP16

In [5]:
print(f"Loading {MODEL_NAME} with {NUM_LABELS}-class head...")
t0 = time.time()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    torch_dtype=torch.float32,  # Load in fp32, Trainer handles the rest
)
model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

Loading Qwen/Qwen2.5-0.5B with 113-class head...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded in 9.8s
Parameters: 494,134,016
VRAM used: 1.8 GB


## Apply LoRA

In [6]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    modules_to_save=["score"],  # Save the classification head
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total:     {total_params:,}")
print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

Total:     496,397,952
Trainable: 2,263,936 (0.46%)
VRAM: 1.8 GB


## Tokenize

In [7]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

print("Tokenizing...")
train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text", "label_name"])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=["text", "label_name"])
train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")

lengths = [len(x["input_ids"]) for x in train_tok]
print(f"Token lengths: min={min(lengths)}, median={sorted(lengths)[len(lengths)//2]}, max={max(lengths)}")

Tokenizing...


Map:   0%|          | 0/57846 [00:00<?, ? examples/s]

Map:   0%|          | 0/6430 [00:00<?, ? examples/s]

Token lengths: min=2, median=63, max=128


## Train

In [8]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)

training_args = TrainingArguments(
    output_dir="qwen05b_cls_output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=(AMP_DTYPE == "fp16"),
    bf16=(AMP_DTYPE == "bf16"),
    gradient_checkpointing=False,
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

steps_per_epoch = len(train_tok) // (BATCH_SIZE * GRADIENT_ACCUMULATION)
print(f"Training: {NUM_EPOCHS} epochs, {steps_per_epoch} steps/epoch")
t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"\nDone in {train_time:.0f}s ({train_time/60:.1f} min)")
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

Training: 3 epochs, 1807 steps/epoch


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.516327,1.654359,0.538880,0.201172
2,1.408565,1.544793,0.567807,0.220643
3,1.027827,1.610023,0.569673,0.239799



Done in 3575s (59.6 min)
Peak VRAM: 7.7 GB


## Save the trained model

Save the LoRA adapter + classification head so we can reload without retraining.

In [9]:
SAVE_DIR = Path("qwen05b_cls_trained")
SAVE_DIR.mkdir(exist_ok=True)

print(f"Saving model to {SAVE_DIR}...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved. Contents: {[f.name for f in SAVE_DIR.iterdir()]}")

Saving model to qwen05b_cls_trained...
Saved. Contents: ['adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'README.md', 'chat_template.jinja', 'tokenizer_config.json']


## Final evaluation

In [10]:
print("\nFinal evaluation...")
predictions = trainer.predict(val_tok)
pred_ids = np.argmax(predictions.predictions, axis=-1)
true_ids = predictions.label_ids

acc = accuracy_score(true_ids, pred_ids)
macro_f1 = f1_score(true_ids, pred_ids, average="macro", zero_division=0)
weighted_f1 = f1_score(true_ids, pred_ids, average="weighted", zero_division=0)
per_class_f1 = f1_score(true_ids, pred_ids, average=None, zero_division=0,
                        labels=list(range(NUM_LABELS)))
zero_f1 = int(sum(1 for f in per_class_f1 if f == 0.0))


Final evaluation...


## Inference latency benchmark

In [11]:
print("Benchmarking inference latency...")
model.eval()

# Warmup
warmup_batch = {k: v[:4].to(device) for k, v in next(iter(torch.utils.data.DataLoader(
    val_tok.select(range(4)), batch_size=4, collate_fn=collator))).items()}
with torch.no_grad():
    for _ in range(5):
        _ = model(**warmup_batch)

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Single-example latency
single_latencies = []
for i in range(min(100, len(val_tok))):
    single = collator([val_tok[i]])
    single = {k: v.to(device) for k, v in single.items()}

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        _ = model(**single)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    single_latencies.append(time.time() - t0)

single_median = np.median(single_latencies) * 1000
single_p95 = np.percentile(single_latencies, 95) * 1000

# Batch throughput (batch_size=32)
batch_latencies = []
batch_loader = torch.utils.data.DataLoader(
    val_tok.select(range(min(640, len(val_tok)))),
    batch_size=32, collate_fn=collator,
)
for batch in batch_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        _ = model(**batch)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    batch_latencies.append(time.time() - t0)

batch_median = np.median(batch_latencies) * 1000
throughput = 32 / np.median(batch_latencies)

# Time to process 64K examples
time_64k_sec = 64000 / throughput
time_64k_min = time_64k_sec / 60

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"\n{'=' * 60}")
print(f"INFERENCE LATENCY — {gpu_name}")
print(f"{'=' * 60}")
print(f"  Single example:  {single_median:.1f} ms (median), {single_p95:.1f} ms (p95)")
print(f"  Batch (32):      {batch_median:.1f} ms/batch, {throughput:.0f} examples/sec")
print(f"  64K examples:    {time_64k_min:.1f} minutes")

Benchmarking inference latency...

INFERENCE LATENCY — Tesla T4
  Single example:  53.2 ms (median), 61.4 ms (p95)
  Batch (32):      266.5 ms/batch, 120 examples/sec
  64K examples:    8.9 minutes


In [12]:
print(f"\n{'=' * 60}")
print(f"RESULTS — {MODEL_NAME} LoRA cls head ({NUM_EPOCHS} epochs)")
print(f"{'=' * 60}")
print(f"  Accuracy:       {acc:.4f} ({acc*100:.1f}%)")
print(f"  Macro F1:       {macro_f1:.4f}")
print(f"  Weighted F1:    {weighted_f1:.4f}")
print(f"  Zero-F1 classes: {zero_f1}/{NUM_LABELS}")
print(f"  Train time:     {train_time:.0f}s ({train_time/60:.1f} min)")
print(f"  LoRA params:    {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"  Inference:      {single_median:.1f} ms/example, {throughput:.0f} examples/sec")
print(f"  64K complaints: {time_64k_min:.1f} minutes")

print(f"\n{'=' * 60}")
print("COMPARISON")
print(f"{'=' * 60}")
print(f"{'Model':<45} {'Acc':>6} {'MacF1':>6} {'ms/ex':>7} {'Time':>8}")
print("-" * 75)
print(f"{'TF-IDF + LogReg':<45} {'54.2%':>6} {'0.132':>6} {'<1':>7} {'30s':>8}")
print(f"{'ModernBERT-base full FT (3ep)':<45} {'56.6%':>6} {'0.209':>6} {'~3':>7} {'32min':>8}")
print(f"{'ModernBERT-base LoRA (3ep)':<45} {'56.4%':>6} {'0.211':>6} {'~3':>7} {'32min':>8}")
print(f"{'Opus 4.6 zero-shot':<45} {'44.0%':>6} {'0.174':>6} {'2300':>7} {'—':>8}")
print(f"{'Qwen2.5-0.5B LoRA cls ({NUM_EPOCHS}ep)':<45} {f'{acc*100:.1f}%':>6} {f'{macro_f1:.3f}':>6} {f'{single_median:.0f}':>7} {f'{train_time/60:.0f}min':>8}")


RESULTS — Qwen/Qwen2.5-0.5B LoRA cls head (3 epochs)
  Accuracy:       0.5697 (57.0%)
  Macro F1:       0.2398
  Weighted F1:    0.5492
  Zero-F1 classes: 37/113
  Train time:     3575s (59.6 min)
  LoRA params:    2,263,936 (0.46%)
  Inference:      53.2 ms/example, 120 examples/sec
  64K complaints: 8.9 minutes

COMPARISON
Model                                            Acc  MacF1   ms/ex     Time
---------------------------------------------------------------------------
TF-IDF + LogReg                                54.2%  0.132      <1      30s
ModernBERT-base full FT (3ep)                  56.6%  0.209      ~3    32min
ModernBERT-base LoRA (3ep)                     56.4%  0.211      ~3    32min
Opus 4.6 zero-shot                             44.0%  0.174    2300        —
Qwen2.5-0.5B LoRA cls ({NUM_EPOCHS}ep)         57.0%  0.240      53    60min


## Save results

In [13]:
results = {
    "model": MODEL_NAME,
    "method": "LoRA_classification_head_fp16",
    "num_classes": NUM_LABELS,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "max_seq_length": MAX_SEQ_LENGTH,
    "total_params": total_params,
    "trainable_params": trainable_params,
    "trainable_pct": round(100 * trainable_params / total_params, 2),
    "accuracy": round(acc, 4),
    "macro_f1": round(macro_f1, 4),
    "weighted_f1": round(weighted_f1, 4),
    "zero_f1_classes": zero_f1,
    "train_time_sec": round(train_time, 0),
    "inference_ms_per_example": round(single_median, 1),
    "throughput_examples_per_sec": round(throughput, 1),
    "time_64k_minutes": round(time_64k_min, 1),
    "inference_gpu": gpu_name,
    "amp_dtype": AMP_DTYPE,
}
with open("qwen05b_cls_head_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved qwen05b_cls_head_results.json")

Saved qwen05b_cls_head_results.json
